In [ ]:
# 실습 준비 — 07주차 연속형 확률변수
# 이 셀을 먼저 한 번 실행하세요. 데이터가 없으면 아래 셀들이 전부 실패합니다.
import os, pathlib, urllib.request

BASE = "https://raw.githubusercontent.com/aprilslab/statistics-lab/main/data/"
FILES = []

pathlib.Path("data").mkdir(exist_ok=True)
for name in FILES:
    for dest in (pathlib.Path(name), pathlib.Path("data") / name):
        if not dest.exists():
            urllib.request.urlretrieve(BASE + name, dest)

# '../data/x.csv' 로 읽는 노트북 대응 — 상위 폴더에도 같은 data/ 를 걸어둔다.
# 절대경로(/data)로 박으면 cwd 가 /content 가 아닐 때 깨지므로 상대경로로 건다.
try:
    parent = pathlib.Path("..") / "data"
    if not parent.exists():
        os.symlink(pathlib.Path("data").resolve(), parent)
except OSError:
    pass

print("준비 완료:", ", ".join(FILES) if FILES else "(내려받을 데이터 없음)")


# 연속형 확률변수

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%precision 3
%matplotlib inline

In [ ]:
from scipy import integrate
import warnings

# 적분에 관한 warning을 출력하지 않도록 한다
warnings.filterwarnings('ignore',
                        category=integrate.IntegrationWarning)

## 1차원 연속형 확률변수

### 1차원 연속형 확률변수의 정의
* 확률변수가 취할 수 있는값은 구간 $[a,b]$
* 확률밀도함수(PDF):
  * 확률변수 X가 $x_0 \le X \le x_1$ 구간에 들어갈 확률 $P(x_0 \le X \le x_1)$

$$
P(x_0 \le X \le x_1) = \int_{x_0}^{x_1} f(x) \, dx
$$

In [ ]:
x_range = np.array([0, 1])

불공정한 룰렛의 확률 밀도함수
$$
f(x) = \left\{
\begin{array}{ll}
2x && (0 \le x \le 1) \\
0 && (otherwise)
\end{array}
\right.
$$

In [ ]:
def f(x):
    if x_range[0] <= x <= x_range[1]:
        return 2 * x
    else:
        return 0

In [ ]:
X = [x_range, f]

In [ ]:
xs = np.linspace(x_range[0], x_range[1], 100)

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)

ax.plot(xs, [f(x) for x in xs], label='f(x)', color='gray')
ax.hlines(0, -0.2, 1.2, alpha=0.3)
ax.vlines(0, -0.2, 2.2, alpha=0.3)
ax.vlines(xs.max(), 0, 2.2, linestyles=':', color='gray')

# 0.4부터 0.6 까지 x좌표를 준비
xs = np.linspace(0.4, 0.6, 100)
# xs의 범위로 f(x)와 x축으로 둘러싸인 영역을 진하게 칠함
ax.fill_between(xs, [f(x) for x in xs], label='prob')

ax.set_xticks(np.arange(-0.2, 1.3, 0.1))
ax.set_xlim(-0.1, 1.1)
ax.set_ylim(-0.2, 2.1)
ax.legend()

plt.show()

$$
P(0.4 \le X \le 0.6) = \int_{0.4}^{0.6} 2x \, dx
$$

In [ ]:
# 첫 번째 인수는 피적분함수、두 번째 인수와 세 번째 인수는 적분 범위
integrate.quad(f, 0.4, 0.6)

### 연속형 확률변수에서의 확률의 성질
* 아래 두 식을 만족해야 함


$$
f(x) \le 0\\
\int_{-\infty}^{\infty} f(x)dx = 1
$$

In [ ]:
from scipy.optimize import minimize_scalar

res = minimize_scalar(f)
# 함수의 최솟값은 fun이라는 인스턴스 변수에
res.fun

In [ ]:
integrate.quad(f, -np.inf, np.inf)[0]

### 누적분포함수
$$
F(x) = P(x \le x) = \int_{-\infty}^{\infty} f(x)dx
$$

In [ ]:
def F(x):
    return integrate.quad(f, -np.inf, x)[0]

예)\
룰렛이 0.4부처 0.6 사이의 값을 취할 확률
$$
P(0.4 \le X \le 0.6) = F(0.6) - F(0.4)
$$

In [ ]:
F(0.6) - F(0.4)

In [ ]:
xs = np.linspace(x_range[0], x_range[1], 100)

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)

ax.plot(xs, [F(x) for x in xs], label='F(x)', color='gray')
ax.hlines(0, -0.1, 1.1, alpha=0.3)
ax.vlines(0, -0.1, 1.1, alpha=0.3)
ax.vlines(xs.max(), 0, 1, linestyles=':', color='gray')

ax.set_xticks(np.arange(-0.1, 1.2, 0.1))
ax.set_xlim(-0.1, 1.1)
ax.set_ylim(-0.1, 1.1)
ax.legend()

plt.show()

### 확률변수의 변환
예) \
룰렛의 예에 나오는 값에 2를 곱하고 3을 더한 $Y = 2X + 3$ 일 때, 확률변수 Y의 밀도함수 $g(y)$
$$
g(y) = \left\{
\begin{array}{ll}
\frac{(y-3)}{2} && (3 \le y \le 5) \\
0 && (otherwise)
\end{array}
\right.
$$

\
(누적)분포함수 $G(y)$
$$
G(y) = P(Y \le y) = \int_{-\infty}^{\infty} g(y)dy
$$

In [ ]:
y_range = [3, 5]

def g(y):
    if y_range[0] <= y <= y_range[1]:
        return (y - 3) / 2
    else:
        return 0

def G(y):
    return integrate.quad(g, -np.inf, y)[0]

In [ ]:
ys = np.linspace(y_range[0], y_range[1], 100)

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)

ax.plot(ys, [g(y) for y in ys],
        label='g(y)', color='gray')
ax.plot(ys, [G(y) for y in ys],
        label='G(y)', ls='--', color='gray')
ax.hlines(0, 2.8, 5.2, alpha=0.3)
ax.vlines(ys.max(), 0, 1, linestyles=':', color='gray')

ax.set_xticks(np.arange(2.8, 5.2, 0.2))
ax.set_xlim(2.8, 5.2)
ax.set_ylim(-0.1, 1.1)
ax.legend()

plt.show()

### 1차원 연속형 확률분포의 지표

#### 평균

$$
\mu = E(X) = \int_{-\infty}^{\infty} xf(x)dx
$$

In [ ]:
def integrand(x):
    return x * f(x)

integrate.quad(integrand, -np.inf, np.inf)[0]

### 연속형 확률변수의 기댓값
$$
E(g(X)) = \int_{-\infty}^{\infty} g(x)f(x)dx
$$

In [ ]:
def E(X, g=lambda x: x):
    x_range, f = X
    def integrand(x):
        return g(x) * f(x)
    return integrate.quad(integrand, -np.inf, np.inf)[0]

In [ ]:
E(X)

예)\
확률변수 $X$를 $2X+3$으로 변환한 확률변수 Y의 기댓값
$$
E(Y) = E(2X+3) = \int_{-\infty}^{\infty}(2x+3)f(x)dx
$$

In [ ]:
E(X, g=lambda x: 2*x+3)

In [ ]:
E(X, g=lambda x: 2*x+3)

In [ ]:
2 * E(X) + 3

#### 분산
$$
\sigma^2 = V(X) = \int_{-\infty}^{\infty}(x-\mu)^2f(x)dx
$$

In [ ]:
mean = E(X)
def integrand(x):
    return (x - mean) ** 2 * f(x)

integrate.quad(integrand, -np.inf, np.inf)[0]

예)\
확률변수 $X$를 $2X+3$으로 변환한 확률변수 Y
$$
V(Y) = V(2X+3) = \int_{-\infty}^{\infty}((2x+3)-\mu)^2f(x)dx \\
\text{단,} \mu = E(2X+3)
$$

#### 연속형 확률변수의 분산
$$
V(g(X)) = \int_{-\infty}^{\infty} (g(x) - E(g(X)))^2f(x)dx
$$

In [ ]:
def V(X, g=lambda x: x):
    x_range, f = X
    mean = E(X, g)
    def integrand(x):
        return (g(x) - mean) ** 2 * f(x)
    return integrate.quad(integrand, -np.inf, np.inf)[0]

In [ ]:
V(X)

In [ ]:
V(X, lambda x: 2*x + 3)

In [ ]:
2**2 * V(X)

## 2차원 연속형 확률분포

### ２차원 연속형 확률변수의 정의

### 결합확률밀도함수
* 2차원 연속형 확률변수$(X,Y)$가 취할 수 있는 값을 조합을 정의역으로 하는 함수 $f(x,y)$
* $x_0 \le X \le x_1$및 $y_0 \le Y \le y_1$이 되는 확률
\
$$
P(x_0 \le X \le x_1,y_0 \le Y \le y_1) = \int_{x_0}^{x_1}\int_{y_0}^{y_1}f(x,y)dxdy
$$

\
예)\
불공정한 룰렛 A,B
* A의 값과 B의 값을 더한 것을 확률변수 X
* A의 값이 확률변수 Y
* 확률변수 $(X,Y)$의 정의역 $(0 \le X \le 2,0 \le Y \le 1)$
* 결합확률밀도함수
\
$$
f(x,y) = \left\{
\begin{array}{ll}
4y(x-y) && (0 \le y \le 1 \,\, \text{및} \,\, 0 \le x-y \le 1) \\
0 && (otherwise)
\end{array}
\right.
$$

### 확률의 성질
* 2차원 연속형 확률변수는 확률의 성질로 다음의 두가지를 만족해야함

\
$$
f(x,y) \le 0
$$
\
$$
\int_{-\infty}^{\infty}\int_{-\infty}^{\infty} f(x,y) = 1
$$

In [ ]:
x_range = [0, 2]
y_range = [0, 1]

In [ ]:
def f_xy(x, y):
    if 0 <= y <= 1 and 0 <= x - y <= 1:
        return 4 * y * (x - y)
    else:
        return 0

In [ ]:
XY = [x_range, y_range, f_xy]

In [ ]:
xs = np.linspace(x_range[0], x_range[1], 200)
ys = np.linspace(y_range[0], y_range[1], 200)
pd = np.array([[f_xy(x, y) for y in ys] for x in xs])

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111)

c = ax.pcolor(pd)
ax.set_xticks(np.linspace(0, 200, 3), minor=False)
ax.set_yticks(np.linspace(0, 200, 3), minor=False)
ax.set_xticklabels(np.linspace(0, 2, 3))
ax.set_yticklabels(np.linspace(0, 1, 3))
ax.invert_yaxis()
ax.xaxis.tick_top()
fig.colorbar(c, ax=ax)
plt.show()

In [ ]:
# 첫 번째 인수는 피적분함수、두 번째 인수는 x의 적분구간과 y의 적분구간
integrate.nquad(f_xy,
                [[-np.inf, np.inf],
                 [-np.inf, np.inf]])[0]

### 주변확률밀도함수
* 확률변수 $(X,Y)$중 확률변수 $X$만의 밀도함수를알고 싶을 때
\
$$
f_x(x) = \int_{-\infty}^{\infty}f(x,y)dy
$$

In [ ]:
from functools import partial

def f_X(x):
    return integrate.quad(partial(f_xy, x), -np.inf, np.inf)[0]
def f_Y(y):
    return integrate.quad(partial(f_xy, y=y), -np.inf, np.inf)[0]

In [ ]:
X = [x_range, f_X]
Y = [y_range, f_Y]

In [ ]:
xs = np.linspace(*x_range, 100)
ys = np.linspace(*y_range, 100)

fig = plt.figure(figsize=(12, 4))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)
ax1.plot(xs, [f_X(x) for x in xs], color='gray')
ax2.plot(ys, [f_Y(y) for y in ys], color='gray')
ax1.set_title('X_marginal density function')
ax2.set_title('Y_marginal density function')

plt.show()

### ２차원 연속형 확률변수의 지표

#### 기댓값
* $x$와 밀도함수의 곲을 $x$와 $y$로 적분하여 구함
\
$$
\mu_x = E(X) = \int_{-\infty}^{\infty}\int_{-\infty}^{\infty}xf(x,y)dx \, dy
$$

In [ ]:
def integrand(x, y):
    return x * f_xy(x, y)

integrate.nquad(integrand,
                [[-np.inf, np.inf],
                 [-np.inf, np.inf]])[0]

\
$$
E(g(X,Y)) = \int_{-\infty}^{\infty}\int_{-\infty}^{\infty}g(x,y)f(x,y)dx \, dy
$$

In [ ]:
def E(XY, g):
    x_range, y_range, f_xy = XY
    def integrand(x, y):
        return g(x, y) * f_xy(x, y)

    return integrate.nquad(integrand,
                           [[-np.inf, np.inf],
                            [-np.inf, np.inf]])[0]

In [ ]:
mean_X = E(XY, lambda x, y: x)
mean_X

In [ ]:
mean_Y = E(XY, lambda x, y: y)
mean_Y

In [ ]:
a, b = 2, 3

In [ ]:
E(XY, lambda x, y: a*x + b*y)

In [ ]:
a * mean_X + b * mean_Y

#### 분산
$X$의 분산
\
$$
\sigma^2_x = V(X) = \int_{-\infty}^{\infty}\int_{-\infty}^{\infty}(x-\mu_x)^2f(x,y)dx \, dy
$$

In [ ]:
def integrand(x, y):
    return (x - mean_X)**2 * f_xy(x, y)

integrate.nquad(integrand,
                [[-np.inf, np.inf],
                 [-np.inf, np.inf]])[0]

$g(X,Y)$의 분산
\
$$
V(g(X,Y)) = \int_{-\infty}^{\infty}\int_{-\infty}^{\infty}(g(x,y) - E(g(X,Y)))^2f(x,y)dx \, dy
$$

In [ ]:
def V(XY, g):
    x_range, y_range, f_xy = XY
    mean = E(XY, g)
    def integrand(x, y):
        return (g(x, y) - mean)**2 * f_xy(x, y)

    return integrate.nquad(integrand,
                           [[-np.inf, np.inf],
                            [-np.inf, np.inf]])[0]

In [ ]:
var_X = V(XY, lambda x, y: x)
var_X

In [ ]:
var_Y = V(XY, lambda x, y: y)
var_Y

#### 공분산
\
$$
\sigma_{XY} = Cov(X,Y) = \int_{-\infty}^{\infty}\int_{-\infty}^{\infty}(x-\mu_X)(y-\mu_Y)f(x,y)dx \, dy
$$

In [ ]:
def Cov(XY):
    x_range, y_range, f_xy = XY
    mean_X = E(XY, lambda x, y: x)
    mean_Y = E(XY, lambda x, y: y)
    def integrand(x, y):
        return (x-mean_X) * (y-mean_Y) * f_xy(x, y)

    return integrate.nquad(integrand,
                           [[-np.inf, np.inf],
                            [-np.inf, np.inf]])[0]

In [ ]:
cov_xy = Cov(XY)
cov_xy

In [ ]:
V(XY, lambda x, y: a*x + b*y)

In [ ]:
a**2 * var_X + b**2 * var_Y + 2*a*b * cov_xy

In [ ]:
cov_xy / np.sqrt(var_X * var_Y)